---슈도 코드  
크로마 DB 접속  
크로마 DB에 테이블 생성 
트레이닝 데이터 중 질문만 백터화하고 나머지는 메타 데이터로 입력(시스템 프롬프트는 제외)  
오픈 ai LLM 연결  
벡터 유사도 검색 테스트  
시간 남으면 평가 데이터로 질문 정확도 테스트(평가 데이터 중 골라서 질문을 넣기or트레이닝 데이터랑 유사한 질문을 조원들끼리 나눠서 생성하고 알맞은 답변을 골라놓기)  


In [2]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

c:\Users\Playdata\Desktop\mle-01-p1-team2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2041.08it/s]


임베딩 모델 준비 완료 — 벡터 차원: 768


In [17]:
df = pd.read_csv('../data/training/df.csv')
print(df)

       Unnamed: 0 meta.lifeCycle meta.department meta.disease  \
0               0             자견              내과           기타   
1               1             성견              내과           기타   
2               2             성견              내과           기타   
3               3            노령견              내과           기타   
4               4             성견              내과           기타   
...           ...            ...             ...          ...   
19201       19201            노령견             피부과           기타   
19202       19202             성견             피부과           기타   
19203       19203            노령견             피부과           기타   
19204       19204             성견             피부과           기타   
19205       19205            노령견             피부과           기타   

                                          qa.instruction  \
0               너는 반려견 건강 전문가야. 반려견 보호자의 질문에 전문가로서 답변해줘.   
1        너는 반려견의 건강에 대해 해박한 지식을 가진 전문가야. 보호자의 궁금증을 해소해줘.   
2      너는 반려견의 건강 상태를 정확하게 평가할 수 있는 수의사

In [18]:
df = df.drop(labels='qa.instruction',axis=1)

In [ ]:
print(df.head())

       Unnamed: 0 meta.lifeCycle meta.department meta.disease  \
0               0             자견              내과           기타   
1               1             성견              내과           기타   
2               2             성견              내과           기타   
3               3            노령견              내과           기타   
4               4             성견              내과           기타   
...           ...            ...             ...          ...   
19201       19201            노령견             피부과           기타   
19202       19202             성견             피부과           기타   
19203       19203            노령견             피부과           기타   
19204       19204             성견             피부과           기타   
19205       19205            노령견             피부과           기타   

                                                qa.input  \
0      저희 집에서 기르고 있는 것으로 강아지는 제 팔뚝 정도 되는 작은 크기의 반려견입니...   
1      저희 집에서 기르고 있는 진돗개가 최근 설사를 하는 증상을 보이고 있습니다. 특히 ...   
2      항문낭을 관리하지 않다가 몇일 전, 제가 앉아 있는 동안 

In [7]:
print(df.shape)
print(df['meta.disease'].value_counts())

(19206, 6)
meta.disease
기타           14774
중성화수술          579
무릎뼈 탈구         517
중독             414
골절             364
심장사상충          352
자궁축농증          329
외이염            288
위장관폐색          228
아토피성 피부염       224
방광염            200
결막염            134
부신피질기능항진증      117
제3안검탈출증         96
방광결석            81
유루증             81
급성신부전           68
전방십자인대파열        67
녹내장             49
치은염             44
복막염             30
혈소판감소증          29
단두종증후군          22
추간판질환           20
유선종양            13
무릎뼈탈구           12
건성각결막염          10
치근단농양           10
췌장염              9
담낭정액낭종           9
만성신부전            8
피부사상균증           7
백내장              4
빈혈               2
항문낭염             2
저혈당              2
기관지염             2
흉수               2
폐렴               1
위장관출혈            1
식도이물             1
각막궤양             1
장내이물             1
고관절이형증           1
Name: count, dtype: int64


In [20]:
df.rename(columns = {'Unnamed: 0' : 'ids'}, inplace = True)

In [23]:
df.head()

,ids,meta.lifeCycle,meta.department,meta.disease,qa.input,qa.output
0,0,자견,내과,기타,저희 집에서 기르고 있는 것으로 강아지는 제 팔뚝 정도 되는 작은 크기의 반려견입니...,"질문하신 내용을 잘 확인하였습니다. 첫째로, 일반적으로 개의 분만 시 태어나는 새끼..."
1,1,성견,내과,기타,저희 집에서 기르고 있는 진돗개가 최근 설사를 하는 증상을 보이고 있습니다. 특히 ...,"보내주신 질문을 면밀히 검토한 결과, 현재 강아지께서 수양성 설사를 경험하고 있는 ..."
2,2,성견,내과,기타,"항문낭을 관리하지 않다가 몇일 전, 제가 앉아 있는 동안 제 하체에 강아지가 앉아 ...","항문낭을 오랫동안 관리하지 않다가 항문낭을 짜주게 되면, 항문낭과 주변 조직이 자극..."
3,3,노령견,내과,기타,현재 저희 강아지가 파보장염에 걸린 것으로 의심되고 있습니다. 저는 3개월 된 포메...,"말씀하신 대로, 파보 바이러스에 의한 장염은 특히 어린 강아지에게 치명적인 감염성 ..."
4,4,성견,내과,기타,강아지의 심장에 관한 질문이 있습니다. 저희 강아지가 동물병원에서 실시한 종합검사 ...,2.5기 상태라면 약물 치료가 필요할 것으로 판단됩니다. 검사 결과지만으로도 약 처...


In [8]:
# 1) 인덱싱 타임 — 모든 여행지 소개를 임베딩(문서 개수 × 768차원)
doc_texts = df['qa.input'].tolist()
doc_emb = emb_model.encode(doc_texts, normalize_embeddings=True)

print("문서 임베딩 행렬:", doc_emb.shape)

# 1.0 에 아주 가까우면 길이가 1로 잘 맞춰진 것이다
print("첫 문서 벡터의 길이:", round(float(np.linalg.norm(doc_emb[0])), 4))

문서 임베딩 행렬: (19206, 768)
첫 문서 벡터의 길이: 1.0


In [28]:
df['ids'] = df.ids.astype(str)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19206 entries, 0 to 19205
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ids              19206 non-null  object
 1   meta.lifeCycle   19206 non-null  object
 2   meta.department  19206 non-null  object
 3   meta.disease     19205 non-null  object
 4   qa.input         19206 non-null  object
 5   qa.output        19206 non-null  object
dtypes: object(6)
memory usage: 900.4+ KB


In [32]:
print(len(df))
print(len(doc_emb))

19206
19206


In [31]:
# 1) 클라이언트 — 메모리에 살아서 커널을 끄면 사라진다
client = chromadb.PersistentClient(path='./chroma_db')

# 2) 컬렉션 — get_or_create 라 여러 번 실행해도 안전하다
pet_care_col = client.get_or_create_collection(
    'pet_care',
    metadata={'hnsw:space': 'cosine'},   # 거리 기준
)

# 3) 적재 — 네 인자는 같은 위치끼리 한 문서를 이룬다
pet_care_col.add(
    ids=df['ids'].tolist(),
    embeddings=doc_emb,                        # 우리 벡터를 직접 넣는다
    documents=df['qa.input'].tolist(),
    metadatas=[
        {'meta.lifeCycle': n, 'meta.department': r, 'meta.disease': t, 'qa.output': f}
        for n, r, t, f in zip(
            df['meta.lifeCycle'], df['meta.department'], df['meta.disease'], df['qa.output']
        )
    ],
)

print("컬렉션에 저장된 문서 수:", pet_care_col.count())

# 4) get 은 검색이 아니라 id 로 직접 꺼내는 것
peek = pet_care_col.get(ids=['0'])

print('원문:', peek['documents'][0])
print('메타데이터:', peek['metadatas'][0])

InternalError: ValueError: Batch size of 19206 is greater than max batch size of 5461

In [ ]:
import chromadb

# 1) ChromaDB 클라이언트
client = chromadb.PersistentClient(path='./chroma_db')

# 2) 컬렉션
pet_care_col = client.get_or_create_collection(
    'pet_care',
    metadata={'hnsw:space': 'cosine'}
)

# 3) 데이터 나눠서 적재
BATCH_SIZE = 5000
total = len(df)

for start in range(0, total, BATCH_SIZE):

    end = min(start + BATCH_SIZE, total)

    pet_care_col.add(
        ids=df['ids'].iloc[start:end].astype(str).tolist(),

        embeddings=doc_emb[start:end],

        documents=df['qa.input'].iloc[start:end].tolist(),

        metadatas=[
            {
                'meta.lifeCycle': n,
                'meta.department': r,
                'meta.disease': t,
                'qa.output': f
            }
            for n, r, t, f in zip(
                df['meta.lifeCycle'].iloc[start:end],
                df['meta.department'].iloc[start:end],
                df['meta.disease'].iloc[start:end],
                df['qa.output'].iloc[start:end]
            )
        ],
    )

    print(f"{end}/{total}개 저장 완료")


# 4) 저장 결과 확인
print("컬렉션에 저장된 문서 수:", pet_care_col.count())


# 5) ID로 데이터 확인
peek = pet_care_col.get(ids=['0'])

print('원문:', peek['documents'][0])
print('메타데이터:', peek['metadatas'][0])

5000/19206개 저장 완료
10000/19206개 저장 완료
15000/19206개 저장 완료
19206/19206개 저장 완료
컬렉션에 저장된 문서 수: 19206
원문: 저희 집에서 기르고 있는 것으로 강아지는 제 팔뚝 정도 되는 작은 크기의 반려견입니다. 그런데 정확한 시기는 기억나지 않지만, 약 한 달 전 즈음에 같은 아파트에 거주하는 수컷 강아지와 교배를 하였습니다. 교배 이후 약 3주가 지나자 우리 집 강아지가 살이 찌는 모습을 보였습니다. 저는 걱정스러운 부분이 있습니다. 일반적으로 암컷이 수컷보다 약간 더 큰 것이 이상적이 라고 알고 있는 것으로 데, 저희 강아지는 그와 는 정반대의 경우로, 수컷 강아지가 2. 5배에서 3배 정도 더 큰 상황입니다. 이러한 경우, 나중에 분만할 때 어려움이 없을 지 걱정됩니다. 우리 강아지의 크기가 작아서 1마리 정도 임신할 것 같다는 생각이 드는 데, 만약 1마리를 가지게 된다면 분만 과정이 더욱 어려워지지 않을 까 염려됩니다. 언제나 힘이 없고 건강한 편이 아니라 허약한 것 같은 느낌이 드는 데, 혹시 제왕절개를 해야 하는 상황이 라면 우리 강아지가 잘 버텨줄 수 있을 지에 대해 자세히 답변해 주시면 감사하겠습니다.
메타데이터: {'qa.output': '질문하신 내용을 잘 확인하였습니다. 첫째로, 일반적으로 개의 분만 시 태어나는 새끼의 강아지 수는 모견의 크기에 따라 달라지게 됩니다. 대형견일수록 배란되는 난자의 수가 증가 하는 경향이 있습니다. 둘째로, 보통 2살 정도의 경우에 배란되는 난자 수가 가장 많고, 이후에는 점차 감소하는 양상을 보입니다. 아가 의 크기는 아버지와 어머니의 크기에 비례하는 경향이 있습니다. 모견이 4살인 작은 강아지라는 점을 감안할 때, 일반적으로 태어날 가능성이 있는 것으로 새끼의 수는 1~3마리 정도일 것으로 예상됩니다. 만약 아버지가 크다면 태어나는 아가 는 어머니의 복강 크기에 비해 조금 더 큰 경우가 있을 수 있습니다. 일반적으로 정상적인 경우 3~4마리의 새끼를